# 实验5.3 ONNX模型到OM模型转换与验证云沙箱实验

> **课程**：CANN 昇腾人工智能应用开发  
> **模型**：SimpleCNN（两层卷积 + 两层全连接）  
> **数据集**：MNIST 手写数字  
> **硬件平台**：ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB  
> **软件环境**：CANN 9.0.0, Python 3.11, A2-arm

---

## 实验总览

本实验在 **GitCode 昇腾 910B3 云沙箱**环境中，以 SimpleCNN（MNIST 手写数字识别）为载体，完整走通模型部署的标准化链路：**ONNX 模型 → ATC 编译生成 OM → AscendCL 接口 NPU 推理 → 性能评估**。实验同时提供 Python 推理程序，并对比 FP32 全精度、剪枝等多种模型格式的部署效果。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">内容</th>
<th style="text-align: left;">产出</th>
</tr>
<tr>
<td style="text-align: left;"><strong>Part 1</strong></td>
<td style="text-align: left;">SimpleCNN 模型构建与 MNIST 训练</td>
<td style="text-align: left;"><code>simplecnn_mnist_fp32.pth</code></td>
</tr>
<tr>
<td style="text-align: left;"><strong>Part 2</strong></td>
<td style="text-align: left;">PyTorch → ONNX 模型导出与验证</td>
<td style="text-align: left;"><code>simplecnn_mnist_fp32.onnx</code></td>
</tr>
<tr>
<td style="text-align: left;"><strong>Part 3</strong></td>
<td style="text-align: left;">模型优化：剪枝 (Pruning)</td>
<td style="text-align: left;"><code>simplecnn_mnist_pruned.onnx</code></td>
</tr>
<tr>
<td style="text-align: left;"><strong>Part 4</strong></td>
<td style="text-align: left;">ATC 工具：ONNX → OM 模型转换</td>
<td style="text-align: left;"><code>*.om</code> 文件</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Part 5</strong></td>
<td style="text-align: left;">AscendCL 推理验证（CANN Runtime）</td>
<td style="text-align: left;">端侧推理结果</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Part 6</strong></td>
<td style="text-align: left;">综合对比与分析</td>
<td style="text-align: left;">精度/性能/体积对比图表</td>
</tr>
</table>

## 学习目标

1. 理解 CNN 卷积神经网络的基本结构与前向传播
2. 掌握 PyTorch 模型导出 ONNX 的流程与注意事项
3. 理解模型压缩技术：**剪枝**（减少参数）
4. 学会使用 **ATC** 工具完成 ONNX → OM 转换，理解转换中的关键参数
5. 掌握基于 **AscendCL (ACL)** 的端侧推理验证流程
6. 能够对不同优化策略的模型进行量化对比分析

---

## Part 0：环境准备与检查

> 确认所有依赖库可用，检测 NPU/CPU 设备。优先使用本地 MNIST 数据集，避免下载等待。

**环境检查说明**：
- 检查 PyTorch、torch_npu（NPU适配）、torchvision、onnx、onnxruntime、acl（AscendCL）等核心依赖库。
- 自动选择训练设备：优先NPU（npu:0），其次CUDA，最后CPU。
- 后续数据加载将优先检测本地 `data/mnist.npz` 文件，若存在则直接加载，避免重复下载浪费时间；若本地不存在或损坏则从公共数据集自动下载。

In [ ]:
import os, sys, io, time, copy, warnings, subprocess
warnings.filterwarnings('ignore')

if sys.platform == 'win32':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print('=' * 60)
print('环境检查')
print('=' * 60)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
print(f'[OK] PyTorch {torch.__version__}')

HAS_NPU = False
try:
    import torch_npu
    if torch.npu.is_available():
        HAS_NPU = True
        print(f'[OK] torch_npu loaded, NPU: {torch.npu.get_device_name(0)}')
    else:
        print('[INFO] torch_npu loaded but NPU not available')
except ImportError:
    print('[SKIP] torch_npu not installed')

import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
print(f'[OK] torchvision {torchvision.__version__}')

HAS_ONNX = False
try:
    import onnx
    HAS_ONNX = True
    print(f'[OK] onnx {onnx.__version__}')
except ImportError:
    print('[INFO] onnx not installed, installing...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnx'], capture_output=True)
    try:
        import onnx
        HAS_ONNX = True
        print(f'[OK] onnx {onnx.__version__} installed')
    except ImportError:
        print('[SKIP] onnx install failed')

HAS_ORT = False
try:
    import onnxruntime as ort
    HAS_ORT = True
    print(f'[OK] onnxruntime {ort.__version__}')
except ImportError:
    print('[INFO] onnxruntime not installed, installing...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnxruntime'], capture_output=True)
    try:
        import onnxruntime as ort
        HAS_ORT = True
        print(f'[OK] onnxruntime {ort.__version__} installed')
    except ImportError:
        print('[SKIP] onnxruntime install failed')

HAS_ACL = False
try:
    import acl
    HAS_ACL = True
    print('[OK] acl (AscendCL) loaded')
except ImportError:
    print('[SKIP] acl not installed')

if HAS_NPU:
    device = torch.device('npu:0')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'\n[*] Training device: {device}')
print('=' * 60)

In [ ]:
CONFIG = {
    'data_dir': 'data',
    'npz_file': 'mnist.npz',
    'model_dir': 'models',
    'output_dir': 'output',
    'batch_size': 256,
    'epochs': 3,
    'lr': 1e-3,
    'num_classes': 10,
    'input_shape': (1, 1, 28, 28),
    'soc_version': 'Ascend910B3',
    'prune_amount': 0.30,
    'num_test_samples': 100,
}

# MNIST npz 数据集加载与下载工具
_npz_path = os.path.join(CONFIG['data_dir'], CONFIG['npz_file'])

def load_mnist_npz(npz_path):
    """从 mnist.npz 加载数据，返回 (x_train, y_train, x_test, y_test)"""
    data = np.load(npz_path)
    return data['x_train'], data['y_train'], data['x_test'], data['y_test']

def download_mnist_npz(npz_path):
    """下载 MNIST npz 文件：优先直连地址，失败回退至官方源"""
    import urllib.request
    urls = [
        'https://www.qmpan.com/f/Ek8AF3/mnist.npz',
        'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz',
    ]
    os.makedirs(os.path.dirname(npz_path), exist_ok=True)
    last_err = None
    for url in urls:
        try:
            print(f'[INFO] 正在下载 MNIST: {url}')
            urllib.request.urlretrieve(url, npz_path)
            print(f'[OK] 下载完成: {npz_path} ({os.path.getsize(npz_path)/1024/1024:.1f} MB)')
            return
        except Exception as e:
            print(f'[WARN] 该地址下载失败: {e}')
            last_err = e
    raise last_err

# 检测本地是否已有 MNIST npz 数据集，若不存在或损坏则从网络下载
if os.path.exists(_npz_path):
    try:
        _xt, _yt, _xe, _ye = load_mnist_npz(_npz_path)
        print(f'[OK] 检测到本地 MNIST npz 数据集，跳过下载 '
              f'(train={_xt.shape[0]}, test={_xe.shape[0]})')
    except Exception:
        print('[WARN] 本地 mnist.npz 文件损坏，将重新下载')
        os.remove(_npz_path)
        download_mnist_npz(_npz_path)
else:
    print('[INFO] 本地未找到 MNIST npz 数据集，将自动从网络下载')
    download_mnist_npz(_npz_path)

for d in [CONFIG['model_dir'], CONFIG['output_dir']]:
    os.makedirs(d, exist_ok=True)

print('Experiment config:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

---

## Part 1：SimpleCNN 基础模型构建与 MNIST 训练

### 1.1 CNN 基本原理

**卷积神经网络 (CNN)** 是图像识别的核心网络结构：
- **卷积层 (Conv2d)**：提取局部特征（边缘、纹理等），权值共享减少参数
- **池化层 (MaxPool2d)**：降采样，减少计算量，增强平移不变性
- **全连接层 (Linear)**：将特征映射到类别空间
- **ReLU 激活**：引入非线性，$f(x) = \max(0, x)$

**CNN各层作用详解**：
- **卷积层**：通过滑动窗口提取图像局部特征。每个卷积核是一个特征检测器，不同核检测不同特征（边缘、角点、纹理等）。权值共享机制使同一核在整张图上复用，大幅减少参数量（相比全连接层）。浅层卷积提取低级特征（边缘、线条），深层卷积提取高级语义特征（物体部件、形状）。
- **池化层**：对特征图做下采样（如2×2最大池化将空间尺寸减半），既减少后续计算量，又赋予模型一定的平移不变性（物体在图像中平移1-2像素不影响识别结果）。
- **全连接层**：将卷积/池化提取的特征展平后映射到类别空间（本例为10维，对应数字0-9）。全连接层起到"分类器"的作用，综合各维特征做出最终判断。
- **ReLU激活**：$f(x)=\max(0,x)$，将负值置零、正值保留。ReLU引入非线性（没有它多层网络等价于单层线性变换），且计算简单、梯度不易消失，是深度学习最常用的激活函数。

### 1.2 SimpleCNN 网络结构

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层</th>
<th style="text-align: left;">类型</th>
<th style="text-align: left;">输出尺寸</th>
<th style="text-align: left;">参数量</th>
</tr>
<tr>
<td style="text-align: left;">conv1</td>
<td style="text-align: left;">Conv2d(1→32, 3×3)</td>
<td style="text-align: left;">28×28×32</td>
<td style="text-align: left;">320</td>
</tr>
<tr>
<td style="text-align: left;">pool1</td>
<td style="text-align: left;">MaxPool2d(2)</td>
<td style="text-align: left;">14×14×32</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;">conv2</td>
<td style="text-align: left;">Conv2d(32→64, 3×3)</td>
<td style="text-align: left;">14×14×64</td>
<td style="text-align: left;">18,496</td>
</tr>
<tr>
<td style="text-align: left;">pool2</td>
<td style="text-align: left;">MaxPool2d(2)</td>
<td style="text-align: left;">7×7×64</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;">fc1</td>
<td style="text-align: left;">Linear(3136→128)</td>
<td style="text-align: left;">128</td>
<td style="text-align: left;">401,536</td>
</tr>
<tr>
<td style="text-align: left;">fc2</td>
<td style="text-align: left;">Linear(128→10)</td>
<td style="text-align: left;">10</td>
<td style="text-align: left;">1,290</td>
</tr>
<tr>
<td style="text-align: left;"><strong>合计</strong></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>~421,642</strong></td>
</tr>
</table>

**网络结构详解**：
- **conv1**：输入1通道（灰度图），输出32通道，3×3卷积核，padding=1保持尺寸。参数量=32×(1×3×3+1)=320（权重+偏置）。
- **pool1**：2×2最大池化，空间尺寸减半（28→14），无参数。
- **conv2**：输入32通道，输出64通道，3×3卷积。参数量=64×(32×3×3+1)=18,496。
- **pool2**：2×2最大池化，空间尺寸再减半（14→7）。
- **fc1**：将64×7×7=3136维特征展平后映射到128维。参数量=3136×128+128=401,536，是参数量最大的层。
- **fc2**：将128维映射到10维（对应数字0-9）。参数量=128×10+10=1,290。
- **总参数量约42万**，模型大小约1.6MB（FP32），属于轻量级CNN，适合在教学和实验中快速训练和部署。

In [ ]:
class SimpleCNN(nn.Module):
    '''SimpleCNN: 两层卷积 + 两层全连接, 输入(1,28,28) 输出10类'''
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.relu1(self.conv1(x)))
        x = self.pool(self.relu2(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN(num_classes=10).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal params: {total_params:,}')
print(f'Model size (est): {total_params * 4 / 1024:.2f} KB (FP32)')

dummy_input = torch.randn(1, 1, 28, 28).to(device)
dummy_output = model(dummy_input)
print(f'Input: {dummy_input.shape} -> Output: {dummy_output.shape}')

### 1.3 MNIST 数据集加载

**数据加载说明**：
- 优先从本地 `data/mnist.npz` 文件加载MNIST手写数字数据集（60000训练+10000测试，28×28灰度图像）。
- `mnist.npz` 是 NumPy 压缩存档格式，包含 `x_train`、`y_train`、`x_test`、`y_test` 四个数组，通过 `np.load()` 读取后构建自定义 `MNISTNpzDataset`。
- 若本地不存在 `data/mnist.npz` 或文件损坏，则自动从公共数据集（Google Storage）下载该文件。
- 预处理在自定义 Dataset 中完成：像素值从[0,255]归一化到[0,1]，再使用MNIST全局均值0.1307和标准差0.3081进行标准化。

**预期结果**：训练集60000张，测试集10000张，batch_size=256时训练集235个batch。若本地数据已存在则加载瞬间完成；若需下载则可能需要数十秒至数分钟。

In [ ]:
class MNISTNpzDataset(Dataset):
    '''从 numpy 数组加载 MNIST 数据集，自动归一化'''
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32) / 255.0
        self.images = (self.images - 0.1307) / 0.3081
        self.images = self.images.unsqueeze(1)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

x_train_np, y_train_np, x_test_np, y_test_np = load_mnist_npz(_npz_path)
print(f'Loaded from npz: x_train={x_train_np.shape}, y_train={y_train_np.shape}, '
      f'x_test={x_test_np.shape}, y_test={y_test_np.shape}')

train_dataset = MNISTNpzDataset(x_train_np, y_train_np)
test_dataset = MNISTNpzDataset(x_test_np, y_test_np)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f'Train set: {len(train_dataset)}')
print(f'Test set: {len(test_dataset)}')
print(f'Batch size: {CONFIG["batch_size"]} | Train batches: {len(train_loader)}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    img_show = image.squeeze() * 0.3081 + 0.1307
    ax.imshow(img_show, cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=12)
    ax.axis('off')
plt.suptitle('MNIST Training Samples', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'mnist_samples.png'), dpi=150)
plt.show()

### 1.4 模型训练（NPU 加速）

> 在昇腾 910B3 上训练 SimpleCNN，3 个 epoch 通常 < 30 秒。

**训练过程说明**：
- 使用Adam优化器（lr=0.001）和交叉熵损失函数。
- 每个epoch包含训练阶段（model.train()，启用Dropout）和评估阶段（model.eval()，关闭Dropout）。
- 记算并记录训练/测试的loss和accuracy曲线，用于后续可视化分析。
- 训练3个epoch，MNIST任务较简单，3个epoch通常可达97%以上测试准确率。

**预期结果**：3个epoch后测试准确率约97%-99%，训练总耗时在NPU上约10-30秒。训练曲线应显示loss稳步下降、accuracy稳步上升。若在CPU上训练则耗时可能数分钟。

In [ ]:
def train_model(model, train_loader, test_loader, device, epochs=3, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    print(f'Training {epochs} epochs on {device}...')
    print('-' * 70)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0; correct = 0; total = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward(); optimizer.step()
            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total

        model.eval()
        test_loss = 0.0; correct = 0; total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                test_loss += criterion(output, target).item()
                _, predicted = output.max(1)
                total += target.size(0)
                correct += predicted.eq(target).sum().item()
        test_loss /= len(test_loader)
        test_acc = 100. * correct / total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        print(f'  Epoch {epoch+1}/{epochs}: Train Acc={train_acc:.2f}%, '
              f'Test Acc={test_acc:.2f}%, Test Loss={test_loss:.4f}')
    print('-' * 70)
    return model, history

t0 = time.time()
model = SimpleCNN(num_classes=10).to(device)
model, train_history = train_model(model, train_loader, test_loader, device,
                                   epochs=CONFIG['epochs'], lr=CONFIG['lr'])
print(f'Total training time: {time.time()-t0:.1f}s')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(train_history['train_loss']) + 1)
ax1.plot(epochs_range, train_history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, train_history['test_loss'], 'r-s', label='Test Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(epochs_range, train_history['train_acc'], 'b-o', label='Train Acc')
ax2.plot(epochs_range, train_history['test_acc'], 'r-s', label='Test Acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curve'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle('SimpleCNN MNIST Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curve.png'), dpi=150)
plt.show()

In [ ]:
model_path = os.path.join(CONFIG['model_dir'], 'simplecnn_mnist_fp32.pth')
torch.save(model.state_dict(), model_path)
print(f'FP32 model saved: {model_path} ({os.path.getsize(model_path)/1024:.2f} KB)')

model.eval()
correct = 0; total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
fp32_acc = 100. * correct / total
print(f'FP32 Test Accuracy: {fp32_acc:.2f}% ({correct}/{total})')

---

## Part 2：PyTorch → ONNX 模型导出与验证

**ONNX** (Open Neural Network Exchange) 是框架间的桥梁：
```
PyTorch --> ONNX --> ATC --> OM (Ascend) --> ACL --> inference
```

> ONNX 扮演"中间语言"的角色：PyTorch 只负责把模型"翻译"成 ONNX，ATC 只负责把 ONNX "编译"成 OM，两段解耦、各自独立。

**导出与验证说明**：
- `torch.onnx.export()` 将PyTorch模型追踪并序列化为ONNX格式。`opset_version=11` 指定算子集版本，需确保ATC支持该版本。
- `input_names=['input']` 显式命名输入节点，后续ATC的 `--input_shape` 将引用此名称。
- `dynamic_axes` 指定batch维度为动态，允许推理时使用不同batch大小。
- 导出后用 `onnx.checker.check_model()` 验证模型结构合法性。
- 随后对比PyTorch和ONNX Runtime的推理结果，验证导出无损。

**预期结果**：ONNX文件成功导出（约1100KB），验证通过。PyTorch与ONNX推理准确率一致（差异<0.1%），最大输出差异在1e-6量级（浮点精度误差），说明导出无损。

In [ ]:
onnx_path = os.path.join(CONFIG['model_dir'], 'simplecnn_mnist_fp32.onnx')
if not HAS_ONNX:
    print('[SKIP] onnx not installed, skipping ONNX export')
else:
    model.eval()
    dummy_input = torch.randn(1, 1, 28, 28).to(device)
    torch.onnx.export(
        model, dummy_input, onnx_path,
        opset_version=11, dynamo=False,
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print(f'ONNX exported: {onnx_path} ({os.path.getsize(onnx_path)/1024:.2f} KB)')

    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print('ONNX validation passed!')
    for inp in onnx_model.graph.input:
        print(f'  Input: {inp.name}')
    for out in onnx_model.graph.output:
        print(f'  Output: {out.name}')

In [ ]:
if not (HAS_ONNX and HAS_ORT and os.path.exists(onnx_path)):
    print('[SKIP] onnx/onnxruntime not available or ONNX model not exported, skipping comparison')
    pt_acc = fp32_acc; onnx_acc = fp32_acc
else:
    print('=' * 60)
    print('PyTorch vs ONNX Accuracy Comparison')
    print('=' * 60)
    ort_session = ort.InferenceSession(onnx_path)
    input_name = ort_session.get_inputs()[0].name

    model.eval()
    pt_correct = 0; onnx_correct = 0; total = 0; max_diff = 0.0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            pt_output = model(data)
            pt_pred = pt_output.argmax(dim=1)
            onnx_output = ort_session.run(None, {input_name: data.cpu().numpy()})[0]
            onnx_pred = torch.tensor(onnx_output.argmax(axis=1))
            total += target.size(0)
            pt_correct += pt_pred.eq(target).sum().item()
            onnx_correct += onnx_pred.eq(target.cpu()).sum().item()
            max_diff = max(max_diff, np.abs(pt_output.cpu().numpy() - onnx_output).max())

    pt_acc = 100. * pt_correct / total
    onnx_acc = 100. * onnx_correct / total
    print(f'PyTorch Acc: {pt_acc:.4f}% | ONNX Acc: {onnx_acc:.4f}% | Diff: {abs(pt_acc-onnx_acc):.4f}%')
    print(f'Max output diff: {max_diff:.8f}')
    print('[OK] ONNX matches PyTorch!' if abs(pt_acc - onnx_acc) < 0.1 else '[WARN] Difference detected!')

---

## Part 3：模型优化 — 剪枝

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">原理</th>
<th style="text-align: left;">效果</th>
</tr>
<tr>
<td style="text-align: left;"><strong>剪枝 (Pruning)</strong></td>
<td style="text-align: left;">将不重要的权重置零</td>
<td style="text-align: left;">减少参数量，为压缩打基础</td>
</tr>
</table>

**剪枝技术详解**：
- **L1非结构化剪枝**：对每层权重矩阵计算L1范数（绝对值），将绝对值最小的30%权重置零。这些权重对模型输出贡献最小，置零后精度损失可控。
- **稀疏性（Sparsity）**：剪枝后零值占比。30%剪枝应使稀疏性接近30%。
- **非结构化vs结构化**：非结构化剪枝产生稀疏矩阵（零值分散），不改变模型结构，文件体积不变。结构化剪枝直接移除整个通道/过滤器，减少参数量，模型体积显著缩小，且无需特殊硬件支持即可加速。
- **本实验采用结构化剪枝**：按 L1 范数排序各层输出通道/神经元，移除贡献最小的 30%，物理缩减模型结构（conv1: 32→22, conv2: 64→44, fc1: 128→89），ONNX/OM 文件体积明显减小。

### 3.1 模型剪枝 (L1 结构化剪枝)

> 结构化剪枝按通道/神经元粒度移除不重要的参数，直接缩小模型结构和文件体积。

**预期结果**：剪枝后参数量减少约50%，ONNX/OM文件体积显著缩小。准确率下降通常<1%-2%（30%剪枝率下精度损失可控）。

In [ ]:
def structured_prune_simplecnn(original_model, amount=0.3):
    '''结构化剪枝：按 L1 范数移除不重要的通道/神经元，物理缩减模型结构'''
    conv1_w = original_model.conv1.weight.data.clone()
    conv1_b = original_model.conv1.bias.data.clone()
    conv2_w = original_model.conv2.weight.data.clone()
    conv2_b = original_model.conv2.bias.data.clone()
    fc1_w   = original_model.fc1.weight.data.clone()
    fc1_b   = original_model.fc1.bias.data.clone()
    fc2_w   = original_model.fc2.weight.data.clone()
    fc2_b   = original_model.fc2.bias.data.clone()

    # Step1: 剪枝 conv1 输出通道 (32 -> keep1)
    l1_c1 = conv1_w.abs().sum(dim=(1, 2, 3))
    keep1 = int(conv1_w.size(0) * (1 - amount))
    idx1 = torch.argsort(l1_c1, descending=True)[:keep1].sort()[0]
    conv1_w = conv1_w[idx1]; conv1_b = conv1_b[idx1]
    conv2_w = conv2_w[:, idx1]

    # Step2: 剪枝 conv2 输出通道 (64 -> keep2)
    l1_c2 = conv2_w.abs().sum(dim=(1, 2, 3))
    keep2 = int(conv2_w.size(0) * (1 - amount))
    idx2 = torch.argsort(l1_c2, descending=True)[:keep2].sort()[0]
    conv2_w = conv2_w[idx2]; conv2_b = conv2_b[idx2]
    fc1_w = fc1_w.view(fc1_w.size(0), 64, 7, 7)[:, idx2].reshape(fc1_w.size(0), -1)

    # Step3: 剪枝 fc1 输出神经元 (128 -> keep3)
    l1_f1 = fc1_w.abs().sum(dim=1)
    keep3 = int(fc1_w.size(0) * (1 - amount))
    idx3 = torch.argsort(l1_f1, descending=True)[:keep3].sort()[0]
    fc1_w = fc1_w[idx3]; fc1_b = fc1_b[idx3]
    fc2_w = fc2_w[:, idx3]

    class PrunedCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, keep1, 3, 1, 1)
            self.relu1 = nn.ReLU()
            self.conv2 = nn.Conv2d(keep1, keep2, 3, 1, 1)
            self.relu2 = nn.ReLU()
            self.pool = nn.MaxPool2d(2, 2)
            self.fc1 = nn.Linear(keep2 * 7 * 7, keep3)
            self.relu3 = nn.ReLU()
            self.fc2 = nn.Linear(keep3, 10)
        def forward(self, x):
            x = self.pool(self.relu1(self.conv1(x)))
            x = self.pool(self.relu2(self.conv2(x)))
            x = x.view(-1, keep2 * 7 * 7)
            x = self.relu3(self.fc1(x))
            x = self.fc2(x)
            return x

    dev = next(original_model.parameters()).device
    pruned = PrunedCNN().to(dev)
    pruned.conv1.weight.data = conv1_w; pruned.conv1.bias.data = conv1_b
    pruned.conv2.weight.data = conv2_w; pruned.conv2.bias.data = conv2_b
    pruned.fc1.weight.data = fc1_w;   pruned.fc1.bias.data = fc1_b
    pruned.fc2.weight.data = fc2_w;   pruned.fc2.bias.data = fc2_b
    return pruned, (keep1, keep2, keep3)

base_model = SimpleCNN(num_classes=10).to(device)
base_model.load_state_dict(torch.load(model_path, map_location=device, weights_only=False))

orig_params = sum(p.numel() for p in base_model.parameters())
print(f'Original params: {orig_params:,}')

prune_amount = CONFIG['prune_amount']
pruned_model, (k1, k2, k3) = structured_prune_simplecnn(base_model, amount=prune_amount)
pruned_params = sum(p.numel() for p in pruned_model.parameters())
print(f'Pruned structure: conv1(1->{k1}), conv2({k1}->{k2}), fc1({k2*7*7}->{k3}), fc2({k3}->10)')
print(f'Pruned params: {pruned_params:,} (reduced {100*(1-pruned_params/orig_params):.1f}%)')

def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    return 100. * correct / total

acc_pruned = evaluate_model(pruned_model, test_loader, device)
print(f'Pruned acc: {acc_pruned:.2f}% (drop {fp32_acc-acc_pruned:.2f}%)')

In [ ]:
pruned_onnx_path = os.path.join(CONFIG['model_dir'], 'simplecnn_mnist_pruned.onnx')
pruned_pth_path = os.path.join(CONFIG['model_dir'], 'simplecnn_mnist_pruned.pth')
torch.save(pruned_model.state_dict(), pruned_pth_path)
print(f'Pruned PTH: {pruned_pth_path} ({os.path.getsize(pruned_pth_path)/1024:.2f} KB)')
if not HAS_ONNX:
    print('[SKIP] onnx not installed, skipping pruned ONNX export')
else:
    pruned_model.eval()
    dummy = torch.randn(1, 1, 28, 28).to(device)
    torch.onnx.export(pruned_model, dummy, pruned_onnx_path, opset_version=11, dynamo=False,
                      input_names=['input'], output_names=['output'],
                      dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
    print(f'Pruned ONNX: {pruned_onnx_path} ({os.path.getsize(pruned_onnx_path)/1024:.2f} KB)')
    print(f'FP32 ONNX:   {os.path.getsize(onnx_path)/1024:.2f} KB)')
    print(f'Size reduction: {100*(1-os.path.getsize(pruned_onnx_path)/os.path.getsize(onnx_path)):.1f}%')

---

## Part 4：ATC 模型转换 (ONNX → OM)

**ATC** 将 ONNX 转为昇腾可执行的 OM 格式：
```bash
atc --framework=5 --model=xxx.onnx --output=xxx --soc_version=Ascend910B3 --input_shape="input:1,1,28,28"
```

> **关键参数**：framework=5(ONNX)、input_shape 名称匹配、soc_version 匹配硬件

> **重要教学点**：PyTorch 原生量化（PTQ/QAT）导出的 ONNX 包含 `QuantizeLinear`/`DequantizeLinear` 算子，ATC 不支持这些算子。昇腾端侧量化的正确方案是使用 **AMCT**（Ascend Model Compression Toolkit）。

**ATC转换说明**：
- 本单元格对FP32和剪枝两个ONNX模型分别执行ATC转换，生成对应的OM文件。
- `run_atc()` 函数封装了ATC命令调用，通过 `subprocess.run()` 执行命令行工具。
- 转换成功后输出OM文件大小，转换失败则输出错误信息。
- 转换过程通常需要1-3分钟，期间ATC做算子映射、图融合、内存规划等编译优化。

**预期结果**：两个ONNX模型均成功转换为OM格式。OM文件大小通常比ONNX略小（ATC做了常量折叠和权重优化）。若在非昇腾环境中运行，ATC不可用会输出SKIP提示。

In [ ]:
import subprocess

def run_atc(onnx_path, output_name, soc_version='Ascend910B3', input_shape='1,1,28,28'):
    output_path = os.path.join(CONFIG['model_dir'], output_name)
    cmd = (f'atc --framework=5 --model={onnx_path} --output={output_path} '
           f'--soc_version={soc_version} --input_shape="input:{input_shape}" --log=error')
    print(f'\n{"="*60}')
    print(f'Converting: {os.path.basename(onnx_path)} -> {output_name}.om')
    print(cmd)
    print('='*60)
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        if result.stderr:
            print(f'[STDERR]: {result.stderr[-300:]}')
        om_path = output_path + '.om'
        if os.path.exists(om_path):
            print(f'[OK] OM: {om_path} ({os.path.getsize(om_path)/1024:.2f} KB)')
            return True, om_path
        print('[FAIL/SKIP] OM not generated')
        return False, None
    except Exception as e:
        print(f'[SKIP] {e}')
        return False, None

atc_results = {}
for name, onnx_file, out_name in [
    ('FP32', 'simplecnn_mnist_fp32.onnx', 'simplecnn_mnist_fp32'),
    ('Pruned', 'simplecnn_mnist_pruned.onnx', 'simplecnn_mnist_pruned'),
]:
    ok, om_path = run_atc(os.path.join(CONFIG['model_dir'], onnx_file), out_name)
    atc_results[name] = {'success': ok, 'om_path': om_path}

print('\nATC Summary:')
for name, r in atc_results.items():
    s = 'OK' if r['success'] else 'SKIP'
    sz = f'{os.path.getsize(r["om_path"])/1024:.2f}KB' if r['success'] else 'N/A'
    print(f'  {name:<12} {s:<6} {sz}')

### 4.1 ATC 常见问题

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">问题</th>
<th style="text-align: left;">原因</th>
<th style="text-align: left;">解决</th>
</tr>
<tr>
<td style="text-align: left;"><code>input_shape not match</code></td>
<td style="text-align: left;">动态维度冲突</td>
<td style="text-align: left;">固定 batch 或用 <code>--dynamic_dims</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>op not supported</code></td>
<td style="text-align: left;">算子不支持</td>
<td style="text-align: left;">昇级 CANN 或替换算子</td>
</tr>
<tr>
<td style="text-align: left;"><code>soc_version mismatch</code></td>
<td style="text-align: left;">版本不匹配</td>
<td style="text-align: left;"><code>npu-smi info</code> 查看实际版本</td>
</tr>
<tr>
<td style="text-align: left;"><code>QuantizeLinear not registered</code></td>
<td style="text-align: left;">PyTorch 量化算子 ATC 不支持</td>
<td style="text-align: left;">用 <strong>AMCT</strong> 做昇腾端量化</td>
</tr>
<tr>
<td style="text-align: left;">NumPy 版本冲突</td>
<td style="text-align: left;">ACL 编译于 NumPy 1.x</td>
<td style="text-align: left;"><code>pip install 'numpy<2'</code></td>
</tr>
</table>

**常见问题详解**：
- **input_shape not match**：ONNX导出时用了 `dynamic_axes`（动态batch），但ATC的 `--input_shape` 写死了batch=1。解决方法是在 `--input_shape` 中固定batch，或使用 `--dynamic_dims` 参数指定动态维度。
- **op not supported**：ONNX模型中包含CANN不支持的算子。可能是算子版本过新（opset过高）或算子非标准。解决方法是升级CANN版本或替换为等价的支持算子。
- **QuantizeLinear not registered**：PyTorch的量化感知训练（QAT）或训练后量化（PTQ）会在ONNX中插入QuantizeLinear/DequantizeLinear算子，ATC不支持这些算子。昇腾端量化的正确方案是使用AMCT工具，它直接在OM模型上做量化。
- **NumPy版本冲突**：ACL的Python接口基于NumPy 1.x编译，若环境安装了NumPy 2.x则可能出现API不兼容。通过 `pip install 'numpy<2'` 降级解决。

In [ ]:
if HAS_NPU:
    try:
        r = subprocess.run('npu-smi info', shell=True, capture_output=True, text=True, timeout=10)
        print('Device info:\n', r.stdout)
    except:
        print('Cannot get device info')

om_files = [f for f in os.listdir(CONFIG['model_dir']) if f.endswith('.om')]
print('\nOM models:', om_files if om_files else '(none - run in Ascend env)')

---

## Part 5：AscendCL 推理验证 (CANN Runtime)

AscendCL 推理流程：
```
init ACL -> load OM -> create Dataset -> memcpy H2D -> execute -> memcpy D2H -> release
```

**AscendCL推理流程详解**：
1. **init ACL**：`acl.init()` 初始化运行时，`acl.rt.set_device(0)` + `acl.rt.create_context(0)` 设置设备和上下文。
2. **load OM**：读取OM文件到内存，`acl.mdl.load_from_mem()` 加载模型，`acl.mdl.get_desc()` 查询输入/输出规格。
3. **create Dataset**：在Device侧分配输入/输出内存（`acl.rt.malloc`），创建Dataset对象封装内存缓冲区。
4. **memcpy H2D**：将Host侧预处理后的图像数据通过 `acl.rt.memcpy` 拷贝到Device侧输入内存。
5. **execute**：`acl.mdl.execute()` 执行推理，`acl.rt.synchronize_stream()` 等待完成。
6. **memcpy D2H**：将Device侧输出内存的推理结果拷贝回Host侧，解析为numpy数组。
7. **release**：按逆序释放每次推理的资源（stream→dataset→内存→模型→上下文）。ACL运行时保持初始化状态，不调用 `acl.finalize()`，以免影响后续PyTorch NPU推理。

**预期结果**：FP32 OM模型推理准确率与PyTorch/ONNX一致（约97%-99%），推理速度远快于PyTorch和ONNX（NPU硬件加速）。剪枝OM模型准确率略低（剪枝导致少量精度损失），推理速度因结构缩减而更快。若acl未安装或OM不存在则输出SKIP提示。

In [ ]:
ACL_SUCCESS = 0; ACL_MEM_MALLOC_NORMAL_ONLY = 2
ACL_MEMCPY_HOST_TO_DEVICE = 1; ACL_MEMCPY_DEVICE_TO_HOST = 2

def preprocess_image(image):
    if image.max() > 1.0:
        image = image.astype(np.float32) / 255.0
    else:
        image = image.astype(np.float32)
    image = (image - 0.1307) / 0.3081
    return image.reshape(1, 1, 28, 28)

def infer_om_model(om_path, test_images, test_labels):
    if not HAS_ACL:
        print('[SKIP] ACL not installed')
        return None, None, None
    if not os.path.exists(om_path):
        print(f'[SKIP] OM not found: {om_path}')
        return None, None, None
    print(f'\nLoading OM: {om_path}')
    context, _ = acl.rt.create_context(0)
    with open(om_path, 'rb') as f:
        om_bytes = f.read()
    ptr = acl.util.bytes_to_ptr(om_bytes)
    model_id, _ = acl.mdl.load_from_mem(ptr, len(om_bytes))
    model_desc = acl.mdl.create_desc()
    acl.mdl.get_desc(model_desc, model_id)
    input_size = acl.mdl.get_input_size_by_index(model_desc, 0)
    output_size = acl.mdl.get_output_size_by_index(model_desc, 0)
    if output_size == 0: output_size = 10 * 4
    print(f'  input_size={input_size}, output_size={output_size}')
    in_buf, _ = acl.rt.malloc(input_size, ACL_MEM_MALLOC_NORMAL_ONLY)
    out_buf, _ = acl.rt.malloc(output_size, ACL_MEM_MALLOC_NORMAL_ONLY)
    in_ds = acl.mdl.create_dataset()
    acl.mdl.add_dataset_buffer(in_ds, acl.create_data_buffer(in_buf, input_size))
    out_ds = acl.mdl.create_dataset()
    acl.mdl.add_dataset_buffer(out_ds, acl.create_data_buffer(out_buf, output_size))
    stream, _ = acl.rt.create_stream()
    times = []; preds = []
    for image in test_images:
        x = preprocess_image(image)
        in_ptr = acl.util.numpy_to_ptr(x)
        acl.rt.memcpy(in_buf, input_size, in_ptr, input_size, ACL_MEMCPY_HOST_TO_DEVICE)
        acl.rt.synchronize_stream(stream)
        t0 = time.perf_counter()
        acl.mdl.execute(model_id, in_ds, out_ds)
        t1 = time.perf_counter()
        acl.rt.synchronize_stream(stream)
        buf = acl.mdl.get_dataset_buffer(out_ds, 0)
        data = acl.get_data_buffer_addr(buf)
        actual_size = int(acl.get_data_buffer_size(buf)) or output_size
        num_out = actual_size // 4 if actual_size >= 4 else 10
        output_np = np.zeros(num_out, dtype=np.float32)
        out_ptr = acl.util.numpy_to_ptr(output_np)
        acl.rt.memcpy(out_ptr, actual_size, data, actual_size, ACL_MEMCPY_DEVICE_TO_HOST)
        acl.rt.synchronize_stream(stream)
        preds.append(int(output_np.argmax()))
        times.append((t1 - t0) * 1000)
    acl.rt.destroy_stream(stream)
    acl.mdl.destroy_dataset(in_ds); acl.mdl.destroy_dataset(out_ds)
    acl.rt.free(in_buf); acl.rt.free(out_buf)
    acl.mdl.destroy_desc(model_desc); acl.mdl.unload(model_id)
    acl.rt.destroy_context(context)
    correct = sum(1 for p, l in zip(preds, test_labels) if p == l)
    acc = 100. * correct / len(test_labels)
    print(f'  Acc: {acc:.2f}%, Avg: {np.mean(times):.2f}ms, {1000/np.mean(times):.1f}img/s')
    return preds, times, acc

np.random.seed(42)
_, _, x_test, y_test = load_mnist_npz(_npz_path)
num_samples = CONFIG['num_test_samples']
indices = np.random.choice(len(x_test), num_samples, replace=False)
test_images = [x_test[i] for i in indices]
test_labels = [y_test[i] for i in indices]
print(f'Prepared {num_samples} test images')

In [ ]:
om_inference_results = {}

print('=' * 60 + '\nPyTorch / ONNX Inference\n' + '=' * 60)
model.eval()
pt_times = []; pt_preds = []
for img in test_images:
    x = preprocess_image(img)
    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(torch.tensor(x).to(device)).argmax(dim=1).item()
    pt_times.append((time.perf_counter() - t0) * 1000)
    pt_preds.append(pred)
pt_acc = 100. * sum(1 for p, l in zip(pt_preds, test_labels) if p == l) / num_samples
print(f'PyTorch FP32:  Acc={pt_acc:.2f}%, Avg={np.mean(pt_times):.2f}ms')
om_inference_results['PyTorch'] = {'preds': pt_preds, 'times': pt_times, 'accuracy': pt_acc}

if HAS_ORT and os.path.exists(onnx_path):
    ort_session = ort.InferenceSession(onnx_path)
    input_name = ort_session.get_inputs()[0].name
    onnx_times = []; onnx_preds = []
    for img in test_images:
        x = preprocess_image(img)
        t0 = time.perf_counter()
        output = ort_session.run(None, {input_name: x})[0]
        onnx_times.append((time.perf_counter() - t0) * 1000)
        onnx_preds.append(int(output.argmax()))
    onnx_acc = 100. * sum(1 for p, l in zip(onnx_preds, test_labels) if p == l) / num_samples
    print(f'ONNX FP32:     Acc={onnx_acc:.2f}%, Avg={np.mean(onnx_times):.2f}ms')
    om_inference_results['ONNX'] = {'preds': onnx_preds, 'times': onnx_times, 'accuracy': onnx_acc}
else:
    print('[SKIP] onnxruntime not available or ONNX model not exported')
    om_inference_results['ONNX'] = {'preds': None, 'times': None, 'accuracy': None}

print('\n' + '=' * 60 + '\nOM (ACL) Inference\n' + '=' * 60)
if HAS_ACL:
    acl.init(); acl.rt.set_device(0)
for name, fname in [('FP32', 'simplecnn_mnist_fp32.om'), ('Pruned', 'simplecnn_mnist_pruned.om')]:
    print(f'\n{"="*60}\nOM Inference: {name}\n{"="*60}')
    preds, times, acc = infer_om_model(os.path.join(CONFIG['model_dir'], fname), test_images, test_labels)
    om_inference_results[name] = {'preds': preds, 'times': times, 'accuracy': acc}

---

## Part 6：综合对比与分析

**对比分析说明**：
- 汇总FP32和剪枝模型在PyTorch/ONNX/OM三种格式下的准确率、文件大小和推理耗时。
- 绘制准确率和模型大小对比柱状图，直观展示不同优化策略的效果。
- 分析各优化策略相对基线（FP32 PyTorch）的精度变化和体积缩减比例。

**预期结果与分析**：
- **准确率**：FP32 PyTorch ≈ FP32 ONNX ≈ FP32 OM（转换无损），剪枝模型准确率略降（<0.5%）。
- **文件大小**：PyTorch(.pth) ≈ ONNX(.onnx) ≈ OM(.om)，剪枝模型体积相近（非结构化剪枝不减少参数量，只置零）。
- **推理速度**：OM >> ONNX > PyTorch，OM在NPU上执行最快（硬件加速+编译优化）。
- **关键发现**：ONNX导出无损；剪枝减少有效参数但精度损失可控；ATC转换启用NPU加速；OM推理显著快于其他格式。

In [ ]:
def get_size(path):
    return os.path.getsize(path) / 1024 if os.path.exists(path) else None

results_table = []
for name, fname, acc, fmt in [
    ('FP32', 'simplecnn_mnist_fp32.pth', fp32_acc, 'PyTorch'),
    ('Pruned 30%', 'simplecnn_mnist_pruned.pth', acc_pruned, 'PyTorch'),
    ('FP32', 'simplecnn_mnist_fp32.onnx', fp32_acc, 'ONNX'),
    ('Pruned', 'simplecnn_mnist_pruned.onnx', acc_pruned, 'ONNX'),
]:
    results_table.append({'model': name, 'format': fmt, 'accuracy': acc,
                          'size_kb': get_size(os.path.join(CONFIG['model_dir'], fname))})

for name, fname, acc_key in [('FP32', 'simplecnn_mnist_fp32.om', 'FP32'),
                              ('Pruned', 'simplecnn_mnist_pruned.om', 'Pruned')]:
    om_path = os.path.join(CONFIG['model_dir'], fname)
    om_acc = om_inference_results.get(acc_key, {}).get('accuracy')
    om_times = om_inference_results.get(acc_key, {}).get('times')
    results_table.append({'model': name, 'format': 'OM', 'accuracy': om_acc,
                          'size_kb': get_size(om_path),
                          'avg_time_ms': np.mean(om_times) if om_times else None})

print('=' * 75)
print(f'{"Model":<16} {"Format":<8} {"Accuracy":>10} {"Size(KB)":>12} {"Infer(ms)":>12}')
print('=' * 75)
for r in results_table:
    a = f"{r['accuracy']:.2f}%" if r['accuracy'] is not None else 'N/A'
    s = f"{r['size_kb']:.2f}" if r['size_kb'] is not None else 'N/A'
    t = f"{r.get('avg_time_ms'):.2f}" if r.get('avg_time_ms') is not None else 'N/A'
    print(f"{r['model']:<16} {r['format']:<8} {a:>10} {s:>12} {t:>12}")
print('=' * 75)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pt_r = [r for r in results_table if r['format'] in ('PyTorch', 'ONNX')]
names = [r['model'] + '(' + r['format'] + ')' for r in pt_r]
accs = [r['accuracy'] or 0 for r in pt_r]
sizes = [r['size_kb'] or 0 for r in pt_r]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

ax1 = axes[0]
bars = ax1.bar(range(len(names)), accs, color=colors[:len(names)], alpha=0.8, edgecolor='black')
ax1.set_xticks(range(len(names))); ax1.set_xticklabels(names, rotation=20, ha='right')
ax1.set_ylabel('Accuracy (%)'); ax1.set_title('Accuracy Comparison', fontsize=13)
for b, a in zip(bars, accs):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.1, f'{a:.2f}%', ha='center', fontsize=10)
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
bars = ax2.bar(range(len(names)), sizes, color=colors[:len(names)], alpha=0.8, edgecolor='black')
ax2.set_xticks(range(len(names))); ax2.set_xticklabels(names, rotation=20, ha='right')
ax2.set_ylabel('Size (KB)'); ax2.set_title('Model Size Comparison', fontsize=13)
for b, s in zip(bars, sizes):
    if s: ax2.text(b.get_x()+b.get_width()/2, b.get_height()+5, f'{s:.0f}KB', ha='center', fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('SimpleCNN MNIST - Comprehensive Comparison', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 70)
print('Optimization Analysis')
print('=' * 70)
bl_size = results_table[0]['size_kb']; bl_acc = results_table[0]['accuracy']
print(f'\nBaseline: Acc={bl_acc:.2f}%, Size={bl_size:.2f} KB')
for r in results_table[1:]:
    if r['size_kb'] and r['accuracy']:
        sr = (1 - r['size_kb']/bl_size)*100
        ad = bl_acc - r['accuracy']
        print(f"\n{r['model']} ({r['format']}): Acc={r['accuracy']:.2f}% ({ad:+.2f}%), "
              f"Size={r['size_kb']:.2f}KB ({sr:+.1f}%)")

print('\n' + '=' * 70)
print('Key Findings:')
print('=' * 70)
print('1. ONNX export is lossless (same accuracy as PyTorch)')
print('2. Pruning reduces model size with minimal accuracy drop')
print('3. ATC conversion to OM enables NPU acceleration')
print('4. OM inference is significantly faster than PyTorch/ONNX')
print('5. AscendCL provides complete runtime for OM model execution')

---

## Part 7：手写数字测试集生成与验证

> 从 MNIST 测试集中选取 20 张手写数字图片保存到 `output/test_images/` 目录，并提供单张/批量测试功能，生成可视化结果方便理解。

**测试集说明**：
- 从 MNIST 测试集选取 20 张图片（每种数字 2 张），保存为 PNG 格式。
- 文件命名格式：`test_XX_labelY.png`，XX 为序号，Y 为真实标签。
- `test_single_image(path)` 测试单张图片，`test_all_images(dir)` 测试全部图片。
- 每次测试生成可视化结果图，展示输入图片、预测标签和置信度分布。

In [ ]:
from PIL import Image
import json, glob

test_img_dir = os.path.join(CONFIG['output_dir'], 'test_images')
os.makedirs(test_img_dir, exist_ok=True)

np.random.seed(42)
_, _, x_test_all, y_test_all = load_mnist_npz(_npz_path)

num_test_imgs = 20
selected_indices = []
for digit in range(10):
    digit_indices = np.where(y_test_all == digit)[0]
    selected_indices.extend(np.random.choice(digit_indices, 2, replace=False))
selected_indices = selected_indices[:num_test_imgs]

test_meta = []
for i, idx in enumerate(selected_indices):
    img = x_test_all[idx]
    label = int(y_test_all[idx])
    fname = f'test_{i:02d}_label{label}.png'
    Image.fromarray(img).save(os.path.join(test_img_dir, fname))
    test_meta.append({'file': fname, 'label': label, 'index': int(idx)})

with open(os.path.join(test_img_dir, 'test_meta.json'), 'w') as f:
    json.dump(test_meta, f, indent=2)

print(f'Generated {num_test_imgs} test images in {test_img_dir}/')
for m in test_meta:
    print(f'  {m["file"]} (true label: {m["label"]})')

In [ ]:
def test_single_image(image_path, model, device, save_result=True):
    '''测试单张手写数字图片，返回预测结果并可视化'''
    img = np.array(Image.open(image_path).convert('L'))
    x = preprocess_image(img)
    model.eval()
    with torch.no_grad():
        output = model(torch.tensor(x).to(device))
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]
        pred = int(probs.argmax())
    basename = os.path.basename(image_path)
    true_label = int(basename.split('label')[1].split('.')[0]) if 'label' in basename else None
    correct = (true_label == pred) if true_label is not None else None

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img, cmap='gray')
    ax1.set_title(f'Input: {basename}', fontsize=11)
    ax1.axis('off')
    colors = ['green' if i == pred else 'gray' for i in range(10)]
    ax2.bar(range(10), probs, color=colors, alpha=0.8, edgecolor='black')
    ax2.set_xticks(range(10)); ax2.set_xlabel('Digit'); ax2.set_ylabel('Probability')
    status = 'OK' if correct else 'WRONG' if correct is not None else '?'
    ax2.set_title(f'Prediction: {pred} (true: {true_label}) [{status}]', fontsize=11)
    plt.tight_layout()
    if save_result:
        result_path = image_path.replace('.png', '_result.png')
        plt.savefig(result_path, dpi=150, bbox_inches='tight')
        print(f'Result saved: {result_path}')
    plt.show()
    print(f'Prediction: {pred}, True: {true_label}, Correct: {correct}, Confidence: {probs[pred]*100:.2f}%')
    return pred, probs

def test_all_images(test_dir, model, device):
    '''测试目录下所有测试图片，生成汇总结果'''
    image_files = sorted([f for f in glob.glob(os.path.join(test_dir, 'test_*_label*.png'))
                          if 'result' not in f and 'summary' not in f])
    if not image_files:
        print(f'No test images found in {test_dir}'); return
    print(f'Testing {len(image_files)} images...\n')
    results = []
    for fpath in image_files:
        img = np.array(Image.open(fpath).convert('L'))
        x = preprocess_image(img)
        with torch.no_grad():
            output = model(torch.tensor(x).to(device))
            probs = torch.softmax(output, dim=1).cpu().numpy()[0]
            pred = int(probs.argmax())
        basename = os.path.basename(fpath)
        true_label = int(basename.split('label')[1].split('.')[0])
        correct = (pred == true_label)
        results.append({'file': basename, 'true': true_label, 'pred': pred, 'correct': correct})
        status = 'OK' if correct else 'WRONG'
        print(f'  {basename}: pred={pred}, true={true_label} [{status}] ({probs[pred]*100:.1f}%)')
    num_correct = sum(r['correct'] for r in results)
    print(f'\nAccuracy: {num_correct}/{len(results)} = {100*num_correct/len(results):.1f}%')

    n = len(results); cols = 5; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
    if rows == 1: axes = axes.reshape(1, -1)
    for i, (r, ax) in enumerate(zip(results, axes.flat)):
        img = np.array(Image.open(os.path.join(test_dir, r['file'])).convert('L'))
        ax.imshow(img, cmap='gray')
        color = 'green' if r['correct'] else 'red'
        ax.set_title(f"{r['file']}\npred={r['pred']} true={r['true']}", fontsize=9, color=color)
        ax.axis('off')
    for i in range(n, rows*cols):
        axes.flat[i].axis('off')
    plt.suptitle(f'Test Results: {num_correct}/{n} correct ({100*num_correct/n:.1f}%)', fontsize=14)
    plt.tight_layout()
    summary_path = os.path.join(test_dir, 'test_summary.png')
    plt.savefig(summary_path, dpi=150, bbox_inches='tight')
    print(f'\nSummary saved: {summary_path}')
    plt.show()

print('=== Test single image ===')
_test_files = sorted([f for f in os.listdir(test_img_dir) if f.startswith('test_') and f.endswith('.png') and 'result' not in f and 'summary' not in f])
if _test_files:
    test_single_image(os.path.join(test_img_dir, _test_files[0]), model, device)

print('\n=== Test all images ===')
test_all_images(test_img_dir, model, device)

### 单张图片测试命令

修改下方路径即可测试任意一张生成的测试图片：

```python
# 测试指定图片
test_single_image('output/test_images/test_05_label3.png', model, device)
```

测试全部图片：

```python
# 测试全部图片
test_all_images('output/test_images/', model, device)
```

---

## 关键问题探究

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">问题</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">ONNX 为什么能成为"中间语言"？如果没有 ONNX，N 个框架 × M 个硬件 = N×M 套适配；ONNX 把它降为 N+M</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">dynamic_axes 到底改变了什么？改变的是输入契约还是计算逻辑？为什么转 OM 时 --input_shape 仍需写死？</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">opset 版本对 ATC 转换的影响：过低可能缺算子，过高可能引入不支持的新算子。为什么本实验选 11？</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">结构化剪枝如何减小模型体积？按 L1 范数移除整个通道/神经元，物理缩减网络结构，参数量和文件体积显著降低</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">从 OM 到真实推理还缺什么？ACL 初始化、OM 加载、内存管理、预处理、推理、输出解析</td>
</tr>
</table>

**关键问题解析**：
- **问题1**：ONNX作为标准中间格式，将"N个框架→M个硬件"的N×M适配问题简化为"N个框架→ONNX"+"ONNX→M个硬件"的N+M问题。例如5个框架×4个硬件=20套适配，通过ONNX降为5+4=9套。
- **问题2**：`dynamic_axes` 只改变输入形状的契约（允许推理时传入不同batch），不改变计算逻辑。转OM时 `--input_shape` 需写死是因为ATC编译时需要确定内存分配方案，动态维度需通过 `--dynamic_dims` 单独指定。
- **问题3**：opset 11是较稳定且广泛支持的版本，包含本实验所有算子且ATC完全支持。过低（如opset 7）可能缺少某些算子，过高（如opset 17+）可能引入ATC不支持的新算子。
- **问题4**：结构化剪枝按 L1 范数排序各层输出通道/神经元，移除贡献最小的30%，物理缩减模型结构（如 conv1: 32→22, conv2: 64→44, fc1: 128→89）。参数量减少约50%，ONNX/OM文件体积显著缩小，且无需硬件支持稀疏计算即可加速。
- **问题5**：从OM到真实推理还需：ACL环境初始化、OM模型加载到Device内存、输入预处理（归一化+格式转换）、Host→Device数据拷贝、执行推理、Device→Host结果拷贝、输出后处理（argmax等）、资源释放。

## 常见问题

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">问题</th>
<th style="text-align: left;">原因</th>
<th style="text-align: left;">解决方案</th>
</tr>
<tr>
<td style="text-align: left;"><code>atc: command not found</code></td>
<td style="text-align: left;">CANN 环境未加载</td>
<td style="text-align: left;"><code>source /usr/local/Ascend/ascend-toolkit/set_env.sh</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>soc_version mismatch</code></td>
<td style="text-align: left;">SOC 版本与芯片不匹配</td>
<td style="text-align: left;"><code>npu-smi info</code> 查看实际型号</td>
</tr>
<tr>
<td style="text-align: left;"><code>QuantizeLinear not registered</code></td>
<td style="text-align: left;">PyTorch 量化算子不支持</td>
<td style="text-align: left;">使用 AMCT 做昇腾端量化</td>
</tr>
<tr>
<td style="text-align: left;"><code>No module named acl</code></td>
<td style="text-align: left;">ACL Python 模块未安装</td>
<td style="text-align: left;">确认 CANN 环境已加载</td>
</tr>
<tr>
<td style="text-align: left;">NumPy 版本冲突</td>
<td style="text-align: left;">ACL 编译于 NumPy 1.x</td>
<td style="text-align: left;"><code>pip install 'numpy<2'</code></td>
</tr>
</table>

**常见问题补充说明**：遇到 `atc: command not found` 时，需先执行CANN环境加载脚本 `source /usr/local/Ascend/ascend-toolkit/set_env.sh`，该脚本设置PATH、LD_LIBRARY_PATH等环境变量。`No module named acl` 通常也是环境未加载导致——ACL的Python包路径需通过CANN环境变量配置。`soc_version mismatch` 必须通过 `npu-smi info` 查询实际芯片型号后准确填写，如Ascend910B3、Ascend310B4等。

---

## 总结

本实验完整走通了从训练到部署的全流程：

```
PyTorch训练 → ONNX导出 → 剪枝优化 → ATC转换 → AscendCL推理
```

### 关键收获

1. **ONNX 是跨框架桥梁**：一次导出，多平台部署，N+M 替代 N×M 适配。ONNX将模型表示为标准算子图，解耦训练框架与推理硬件，是部署链路中的关键"中间人"。
2. **结构化剪枝显著减少模型体积**：30% 剪枝后参数量减少约50%，文件体积明显缩小，精度损失可控。结构化剪枝直接移除整个通道/神经元，无需硬件支持稀疏计算即可加速。
3. **ATC 不只是格式转换**：算子映射、图融合、内存优化带来 NPU 加速。ATC将通用ONNX算子图编译为针对昇腾硬件深度优化的指令流，这是OM推理极快的根本原因。
4. **AscendCL 是完整的推理框架**：从初始化到资源释放的完整流程。ACL提供了设备管理、内存管理、模型管理、推理执行等全套API，是端侧部署的执行入口。
5. **OM 推理极快且稳定**：NPU 硬件加速 + 编译优化。预编译消除了运行时开销，NPU专用计算单元提供了极致性能，且耗时稳定（无框架调度不确定性）。

### 产物清单

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">产物</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_fp32.pth</code></td>
<td style="text-align: left;">PyTorch FP32 权重</td>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_fp32.onnx</code></td>
<td style="text-align: left;">ONNX FP32 模型</td>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_pruned.pth</code></td>
<td style="text-align: left;">剪枝 PyTorch 权重（结构缩减）</td>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_pruned.onnx</code></td>
<td style="text-align: left;">剪枝 ONNX 模型（体积更小）</td>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_fp32.om</code></td>
<td style="text-align: left;">昇腾 OM 离线模型</td>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_pruned.om</code></td>
<td style="text-align: left;">剪枝 OM 模型</td>
</tr>
<tr>
<td style="text-align: left;"><code>output/test_images/</code></td>
<td style="text-align: left;">20 张手写数字测试图片及结果</td>
</tr>
</table>

**产物清单说明**：实验产出涵盖从训练到部署各阶段的模型文件。`.pth`是PyTorch训练权重，`.onnx`是跨平台中间格式，`.om`是昇腾离线推理模型。FP32版本是全精度基线，剪枝版本演示了模型优化效果。通过对比这些产物的精度、体积和推理速度，可以全面评估部署链路各环节的作用。

---

## 参考资料

- [昇腾 CANN 文档](https://www.hiascend.com/document)
- [ATC 工具使用指南](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [AscendCL API 参考](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [CANN 开源社区](https://atomgit.com/cann)